# 05 — Feed-Forward Networks in Transformers
### Starter Notebook

**Learning objectives**
- Understand the role of the FFN in a transformer block
- Implement the standard FFN and compare activations (ReLU, GELU, SiLU)
- Implement SwiGLU — the modern choice in Gemma / LLaMA
- Understand why the expansion ratio matters

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(3)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A — The Standard FFN

After attention aggregates information across positions, the FFN processes each position **independently**:

$$\text{FFN}(x) = \text{activation}(x W_1 + b_1)\; W_2 + b_2$$

The hidden dimension is typically $4 \times d_{model}$.

### Exercise A1 — Implement the standard FFN

In [ ]:
class MyFFN(nn.Module):
    """
    Standard two-layer FFN.

    Args:
        d_model  : input/output dimension
        d_ff     : hidden dimension (default 4 * d_model)
        activation: 'relu', 'gelu', or 'silu'
        dropout  : dropout after activation
    """

    def __init__(self, d_model: int, d_ff: int = None, activation: str = 'gelu', dropout: float = 0.0):
        super().__init__()
        d_ff = d_ff or 4 * d_model

        # TODO: define w1, w2, dropout layer
        # Hint: w1 maps d_model → d_ff; w2 maps d_ff → d_model

        self.act = {'relu': F.relu, 'gelu': F.gelu, 'silu': F.silu}[activation]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement forward pass
        pass


ffn = MyFFN(d_model=64, d_ff=256, activation='gelu')
x = torch.randn(2, 10, 64)
out = ffn(x)
if out is not None:
    print(f'FFN output shape: {out.shape}   # expect (2, 10, 64)')
else:
    print('Not implemented yet.')

## Part B — Activation Functions

The original transformer used ReLU. Modern LLMs prefer GELU or SiLU because they are smooth and non-zero for negative inputs.

### Exercise B1 — Plot all three

In [ ]:
x_vals = torch.linspace(-3, 3, 200)

# TODO: compute y_relu, y_gelu, y_silu
activations = {
    'ReLU' : None,   # F.relu(x_vals)
    'GELU' : None,   # F.gelu(x_vals)
    'SiLU' : None,   # F.silu(x_vals)
}

plt.figure(figsize=(8, 4))
for name, y in activations.items():
    if y is not None:
        plt.plot(x_vals, y.detach(), label=name)
plt.axhline(0, color='k', linewidth=0.5)
plt.axvline(0, color='k', linewidth=0.5)
plt.xlabel('x'); plt.ylabel('activation(x)')
plt.title('Activation Functions')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## Part C — SwiGLU

SwiGLU is used in PaLM, LLaMA, and Gemma. It uses a **gating mechanism**: one branch is passed through SiLU, the other is a linear gate.

$$\text{SwiGLU}(x, W, V, W_2) = (\text{SiLU}(xW) \odot xV)\, W_2$$

Because of the multiplication, the hidden dim is typically $\frac{2}{3} \times 4 \times d_{model}$ to keep parameter count equivalent.

### Exercise C1 — Implement SwiGLU

In [ ]:
class MySwiGLU(nn.Module):
    """
    SwiGLU feed-forward layer.

    Three weight matrices: W (gate via SiLU), V (linear path), W2 (output proj).
    """

    def __init__(self, d_model: int, d_ff: int = None):
        super().__init__()
        # Adjust d_ff so parameter count ≈ standard FFN with d_ff = 4 * d_model
        d_ff = d_ff or int(2/3 * 4 * d_model)

        # TODO: define w_gate, w_up, w_down
        # w_gate and w_up both map d_model → d_ff
        # w_down maps d_ff → d_model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: gate = SiLU(x @ w_gate); up = x @ w_up; return (gate * up) @ w_down
        pass


swiglu = MySwiGLU(d_model=64)
out = swiglu(x)
if out is not None:
    print(f'SwiGLU output: {out.shape}   # expect (2, 10, 64)')
    # Check parameter counts are similar
    ffn_params    = sum(p.numel() for p in MyFFN(64, 256).parameters())
    swiglu_params = sum(p.numel() for p in MySwiGLU(64).parameters())
    print(f'FFN params   : {ffn_params}')
    print(f'SwiGLU params: {swiglu_params}  (should be similar)')

## Part D — Library Comparison

In [ ]:
from src.models.feedforward import FeedForward, SwiGLU

ffn_lib    = FeedForward(d_model=64, d_ff=256, activation='gelu')
swiglu_lib = SwiGLU(d_model=64)

x = torch.randn(2, 10, 64)
print(f'FeedForward output: {ffn_lib(x).shape}')
print(f'SwiGLU output     : {swiglu_lib(x).shape}')

# Expansion ratio visualisation
d_model = 512
configs = [
    ('Standard FFN (4x)',    4 * d_model,       'd_ff = 4 × d_model'),
    ('SwiGLU (2/3 × 4x)',   int(2/3 * 4 * d_model), 'd_ff adjusted for 3 matrices'),
]

print(f'\nExpansion ratio comparison (d_model={d_model}):')
for name, d_ff, note in configs:
    params = 2 * d_model * d_ff  # rough count
    print(f'  {name}: d_ff={d_ff:4d}  (~{params/1e6:.2f}M params/layer)  [{note}]')

## Summary

- The FFN processes each token position **independently** — it adds non-linear transformation on top of the attention's information-mixing.
- **ReLU** → original transformer; **GELU** → GPT / BERT; **SiLU/SwiGLU** → LLaMA / Gemma.
- SwiGLU adds a learned gate, improving quality without increasing parameter count.

**Next:** `06_transformer_block_starter.ipynb` — put attention + FFN together into a full transformer block.